# News Topics Pipeline — Step by Step

Braço 03: the frozen text corpus → one **temporal topic model** (tomotopy's DMR,
with the epidemiological period as the document covariate) → the mean topic
mixture of each (epidemiological week, location).

Mirrors `src/pipeline/news/topics/` (spec → steps → transform → integrity). The
hypothesis, the evidence and the counter-evidence are in
[`notes/03-topic-modeling.md`](../notes/03-topic-modeling.md); the methods
write-up and the measured numbers are in
[`src/pipeline/news/README.md`](../src/pipeline/news/README.md) §4.

This is the third of three **competing representations** of the same corpus — not
a layer on top of braços 01 and 02. All three read the same frozen table and
publish the same grain, so the comparison isolates the representation and nothing
else. See [`notes/00-protocolo-comparacao.md`](../notes/00-protocolo-comparacao.md).

Braço 01 asks **how much** the press wrote. Braço 02 asks what a week's prose
*looks like* as a dense vector. This one asks **what** the week was about.

## Setup

In [1]:
import sys
from pathlib import Path

import numpy as np
import polars as pl

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from pipeline.news.topics import steps as topic_steps
from pipeline.news.topics.integrity import run_all as topic_checks
from pipeline.news.topics.spec import (
    COHERENCE, COVARIATE_GRAIN, EW, LOCATION, MODEL_ID, N_ARTICLES, SELECTED,
    SHARE, STOPWORDS_PT, TERM, TOPIC_ID, WEIGHT,
)
from pipeline.news.corpus import scoped_locations, select_articles
from pipeline.news.lexicon import steps as lex_steps
from pipeline.news.text import steps as text_steps

pl.Config.set_tbl_rows(20)
pl.Config.set_fmt_str_lengths(70)

FIXTURES = PROJECT_ROOT / "tests" / "fixtures" / "news"
LEXICON_PATH = PROJECT_ROOT / "data" / "news" / "reference" / "lexico_dengue.yaml"
FU_PATH = PROJECT_ROOT / "data" / "epidemiological" / "br_federative_units.csv"

NEWS_DIR = PROJECT_ROOT / "data" / "pipeline" / "news"
POOLED_PATH = NEWS_DIR / "EBC_dengue_topics_EW.parquet"
META_PATH = NEWS_DIR / "EBC_dengue_topics_meta.parquet"
SWEEP_PATH = NEWS_DIR / "EBC_dengue_topics_sweep.parquet"
SINAN_PATH = PROJECT_ROOT / "data" / "epidemiological" / "SINAN" / "SINAN_dengue_cases.parquet"

FIXTURES.exists(), LEXICON_PATH.exists(), FU_PATH.exists()

(True, True, True)

## Literature checklist (what this notebook must prove)

1. The model is **DMR with the period as covariate** — the mechanism STM extends —
   and it is chosen for reasons this repository can hold: no R runtime, and no
   dependence on braço 02's encoder.
2. `k` is **selected by coherence, never perplexity**, and the *whole sweep* is
   published so the choice can be checked rather than trusted.
3. Tokenization is **frozen and light**: accent folding, a word regex, a stopword
   list. No lemmatizer, no POS tagger, no spaCy.
4. Every topic ships with its **defining terms**, because "topic 7 predicts dengue"
   is not a finding anyone can check.
5. A week's shares form a **distribution**: they add to 1, and a week with no
   article is **absent** rather than a row of zeros.
6. The article set and the geo scoping are **braços 01/02's** — protocol 00 fixes
   them as common ground, so this braço imports them rather than re-deriving them.
7. The fixture is five pages, so it cannot show the real sweep, the real topics or
   the coverage. Those need the real corpus, and the **Optional — the real
   corpus** section below is where they are measured.

## Step 1 — tokenization is frozen and light

`notes/03` proposed spaCy's `pt_core_news` lemmatization. It was set aside: it
would add a several-hundred-megabyte model as the *only* consumer of a new NLP
runtime, in a repository whose other news braço is pure regex.

What replaced it is folding, a `[a-z]+` regex and a frozen stopword list. Folding
first is what lets the list be one flat tuple — `não`/`nao` collapse to one entry.
The list has two deliberate groups, and the second is the interesting one.

In [2]:
probe = "O ministro disse que a dengue avançou em 36.791 casos, segundo o boletim."

print("raw:    ", probe)
print("tokens: ", topic_steps.tokenize(probe))
print()
print(f"{len(STOPWORDS_PT)} stopwords, in two groups:")
print("  function words   ", ", ".join(STOPWORDS_PT[:14]), "…")
print("  news boilerplate ", ", ".join(STOPWORDS_PT[-11:]))
print()
print("Both groups are needed. The second is content words by part of speech, but")
print("in a corpus of attributed news prose they cluster into an 'attribution'")
print("topic that says nothing about dengue.")
print()
print("Domain vocabulary is deliberately NOT in the list — it is what the topics")
print("are supposed to be made of:")
print("  ", topic_steps.tokenize("Casos de dengue e mortes por aedes aegypti"))

raw:     O ministro disse que a dengue avançou em 36.791 casos, segundo o boletim.
tokens:  ['ministro', 'dengue', 'avancou', 'casos', 'boletim']

151 stopwords, in two groups:
  function words    que, com, sem, sob, por, para, pelo, pela, pelos, pelas, dos, das, nos, nas …
  news boilerplate  disse, diz, afirmou, declarou, segundo, informou, explicou, destacou, lembrou, ressaltou, conforme

Both groups are needed. The second is content words by part of speech, but
in a corpus of attributed news prose they cluster into an 'attribution'
topic that says nothing about dengue.

Domain vocabulary is deliberately NOT in the list — it is what the topics
are supposed to be made of:
   ['casos', 'dengue', 'mortes', 'aedes', 'aegypti']


## Step 2 — the model: DMR, with the period as the covariate

`tomotopy`'s DMR conditions topic **prevalence** on a per-document covariate. Here
the covariate is the period each article falls in. LDA — the same training loop
with the metadata dropped — is the baseline, selected by the `family` parameter.

The covariate grain is a **month**, not the weekly grain the tables are keyed by,
and the reason is statistical: 965 weekly levels against ~4.5 articles each would
be an over-parameterised fit, while ~290 monthly levels get ~15. The published
grain stays weekly — the covariate shapes the *fit*, not the output.

The model is an **optional extra**, so this cell uses the real thing when it is
installed and a clearly-labelled stand-in when it is not.

In [3]:
try:
    factory = topic_steps.load_model_factory(family="dmr", seed=42, iterations=200, min_df=1)
    factory(2)  # import smoke test: does the extra actually work?
    REAL_MODEL = True
    print("using the real tomotopy DMR")
except Exception as error:
    import zlib

    class _StandInModel:
        '''Deterministic stand-in for tomotopy's DMR. NOT a real fit.

        It only has to be *some* deterministic function from tokens to a point on
        the simplex, so the selection, pooling and integrity logic below can run
        without the optional dependency.
        '''

        PEAK_K = 3

        def __init__(self, k, family="dmr", seed=42, iterations=200, min_df=1):
            self.k = k
            self.model_id = topic_steps.model_id(family, k, seed)
            self.n_docs = 0

        def fit(self, docs, covariates):
            if len(docs) != len(covariates):
                raise ValueError(f"{len(docs)} docs but {len(covariates)} covariates")
            self.n_docs = len(docs)
            self._docs = [list(d) for d in docs]
            self._vocab = sorted({t for d in self._docs for t in d}) or ["vazio"]
            rows = []
            for doc in self._docs:
                if not doc:
                    rows.append(np.full(self.k, 1.0 / self.k, dtype=np.float32))
                    continue
                rng = np.random.default_rng(zlib.crc32(" ".join(doc).encode("utf-8")))
                w = rng.random(self.k) + 1e-3
                rows.append((w / w.sum()).astype(np.float32))
            self._shares = np.stack(rows)

        def document_shares(self):
            return self._shares

        def topic_words(self, top_n=15):
            size = len(self._vocab)
            take = min(top_n, size)
            return [
                [(self._vocab[(t * 3 + i) % size], 1.0 - 0.05 * i) for i in range(take)]
                for t in range(self.k)
            ]

        def coherence(self, metric="c_v", top_n=10):
            return 1.0 - abs(self.k - self.PEAK_K) / 100.0

        def ll_per_word(self):
            return -8.0 + self.k / 1000.0

    factory = lambda k: _StandInModel(k)
    REAL_MODEL = False
    print(error)
    print(">>> FALLBACK ACTIVE: the topics below are NOT from a real fit <<<")

print(f"\ncovariate grain = {COVARIATE_GRAIN!r}  (the output grain stays weekly)")
print(f"a month bucket  = {topic_steps.covariate_bucket(__import__('datetime').date(2024, 3, 17))!r}")

using the real tomotopy DMR

covariate grain = 'month'  (the output grain stays weekly)
a month bucket  = '2024-03'


## Step 3 — `k` is selected by coherence, not configured

`notes/03` fixes the criterion: **coherence, never perplexity** — likelihood rises
with k almost mechanically while coherence does not. The whole `k_range` is fitted,
`c_v` scores each candidate, and the argmax wins, with ties going to the **smaller**
k so an uninformative plateau resolves to the more parsimonious model.

There is no `topic_count` constant in `spec.py` on purpose: pinning one would make
the selection look like a schema change.

The fixture is five pages, so the sweep here is deliberately tiny — the real sweep
is in the Optional section below.

In [4]:
corpus = text_steps.build(FIXTURES)
lexicon = lex_steps.load_lexicon(LEXICON_PATH)
matcher = lex_steps.load_uf_matcher(FU_PATH)

kept, dropped = select_articles(corpus, lexicon)
docs = topic_steps.tokenize_corpus(kept)
article_ids, ews = topic_steps.article_frame(kept)
covariates = [topic_steps.covariate_bucket(ew, COVARIATE_GRAIN) for ew in ews]

print(f"{corpus.height} fixture pages → {kept.height} kept, {dropped} dropped by the shared gates")
print(f"{len(docs)} documents tokenized, covariates: {sorted(set(covariates))}\n")

sweep = topic_steps.select_k(docs, covariates, (2, 3, 4, 5), factory)

print(topic_steps.sweep_table(sweep))
print(f"\nselected k = {sweep.best_k}  (coherence {sweep.best_coherence:.4f})")

5 fixture pages → 5 kept, 0 dropped by the shared gates
5 documents tokenized, covariates: ['2002-09', '2008-03', '2011-01', '2016-01', '2021-03']

shape: (4, 5)
┌────────────────────────┬──────────┬───────────┬─────────────┬──────────┐
│ model_id               ┆ topic_id ┆ coherence ┆ ll_per_word ┆ selected │
│ ---                    ┆ ---      ┆ ---       ┆ ---         ┆ ---      │
│ str                    ┆ i64      ┆ f32       ┆ f32         ┆ bool     │
╞════════════════════════╪══════════╪═══════════╪═════════════╪══════════╡
│ tomotopy-dmr-k2-seed42 ┆ 2        ┆ 0.510677  ┆ -7.322097   ┆ false    │
│ tomotopy-dmr-k3-seed42 ┆ 3        ┆ 0.673955  ┆ -7.16948    ┆ false    │
│ tomotopy-dmr-k4-seed42 ┆ 4        ┆ 0.710547  ┆ -6.945325   ┆ false    │
│ tomotopy-dmr-k5-seed42 ┆ 5        ┆ 0.727015  ┆ -6.952787   ┆ true     │
└────────────────────────┴──────────┴───────────┴─────────────┴──────────┘

selected k = 5  (coherence 0.7270)


The `sweep_table` is a published artifact, not a log line: it carries the models
that **lost**, with their coherence *and* their `ll_per_word`. That last column is
the one that matters for the argument — it is reported and deliberately **not** used
to choose, so a reader can check that the two criteria disagree.

## Step 4 — the topic → term table

Not optional. A topic index with no readable definition is uninterpretable in a
paper, and "topic 7 predicts dengue" is not a finding anyone can check. Every topic
is published with its defining terms.

In [5]:
model = sweep.best
meta = topic_steps.topic_meta(model, model.model_id, top_n=6)

for topic_id in range(model.k):
    terms = meta.filter(pl.col(TOPIC_ID) == topic_id).sort(WEIGHT, descending=True)[TERM].to_list()
    print(f"  topic {topic_id}: {', '.join(terms)}")

print(f"\n{meta.height} rows = {model.k} topics × {meta.height // model.k} terms")

  topic 0: brasil, rio, forum, discute, atual, macedo
  topic 1: acoes, aegypti, maior, municipio, realizada, planalto
  topic 2: aedes, ministros, padilha, alexandre, reuniao, pais
  topic 3: dengue, combate, saude, agencia, mosquito, brasilia
  topic 4: presidenta, febre, uniao, pede, dilma, alta

30 rows = 5 topics × 6 terms


## Step 5 — pooling to (ew, location)

The same aggregation contract braço 02 uses: every article contributes to **BR**,
and also to each UF it names, so one article can land in several rows of the same
week. The mean is **unweighted**, so each article speaks once regardless of length.

Only observed pairs are emitted. A week with no article is *absent*, not a row of
zeros — a zero vector is not a point in the simplex and would read as "coverage that
says nothing" rather than "no coverage".

In [6]:
pooled = topic_steps.pool_shares(
    article_ids, ews, model.document_shares(), scoped_locations(kept, matcher), model.model_id
)

print(f"{len(article_ids)} articles → {pooled.height} rows over "
      f"{pooled[EW].n_unique()} weeks × {pooled[LOCATION].n_unique()} locations × {model.k} topics")
print(f"shares sum to 1 within each (ew, location): "
      f"{pooled.group_by(EW, LOCATION).agg(pl.col(SHARE).sum())[SHARE].max():.6f} max\n")
pooled.head(6)

5 articles → 30 rows over 5 weeks × 2 locations × 5 topics
shares sum to 1 within each (ew, location): 1.000000 max



ew,location,topic_id,share,n_articles,model_id
date,str,i64,f32,i64,str
2002-09-29,"""BR""",0,0.601438,1,"""tomotopy-dmr-k5-seed42"""
2002-09-29,"""BR""",1,0.002469,1,"""tomotopy-dmr-k5-seed42"""
2002-09-29,"""BR""",2,0.00247,1,"""tomotopy-dmr-k5-seed42"""
2002-09-29,"""BR""",3,0.391153,1,"""tomotopy-dmr-k5-seed42"""
2002-09-29,"""BR""",4,0.00247,1,"""tomotopy-dmr-k5-seed42"""
2002-09-29,"""RJ""",0,0.601438,1,"""tomotopy-dmr-k5-seed42"""


### BR plus each named UF, from one article

`n_articles` repeats across a week's topic rows — it is what makes a share
interpretable, since 0.6 over one article and over forty are different claims.

In [7]:
example = pooled.filter(pl.col(EW) == pooled[EW].min())
print(f"week {example[EW][0]}: {example.height} rows = "
      f"{example[LOCATION].n_unique()} locations × {model.k} topics, "
      f"from {example[N_ARTICLES].max()} article(s)\n")
example.select(LOCATION, TOPIC_ID, SHARE, N_ARTICLES).head(8)

week 2002-09-29: 10 rows = 2 locations × 5 topics, from 1 article(s)



location,topic_id,share,n_articles
str,i64,f32,i64
"""BR""",0,0.601438,1
"""BR""",1,0.002469,1
"""BR""",2,0.00247,1
"""BR""",3,0.391153,1
"""BR""",4,0.00247,1
"""RJ""",0,0.601438,1
"""RJ""",1,0.002469,1
"""RJ""",2,0.00247,1


## Integrity checks — fixture tables

In [8]:
for check in topic_checks(pooled, meta, topic_steps.sweep_table(sweep), FU_PATH):
    print(check)

[PASS] share_schema: 6 columns in published order
[PASS] meta_schema: 4 columns in published order
[PASS] sweep_schema: 5 columns in published order
[PASS] share_unique_keys: 30 rows, all keys distinct
[PASS] meta_unique_keys: 30 rows, all keys distinct
[PASS] weeks_start_on_sunday: all ew dates are Sundays
[PASS] locations_known: 2 known locations
[PASS] shares_sum_to_one: 6 (ew, location) groups, max deviation 5.96e-08
[PASS] shares_in_unit_interval: every share is in [0, 1]
[PASS] finite: all shares and weights are finite
[PASS] article_counts: every row pools at least one article
[PASS] topic_ids_consistent: 5 topics, ids 0..4
[PASS] single_model: all published rows from tomotopy-dmr-k5-seed42
[PASS] sweep_selects_one: k=5 selected on coherence (0.7270; ll_per_word -6.9528)
[PASS] national_row_per_week: 5 weeks all carry BR


`shares_sum_to_one` is the check specific to this braço: a topic mixture is a point
on the simplex, so a week's shares that do not add to 1 mean the pooling is wrong
and the column is not a share.

## Optional — the real corpus

Needs `uv sync --extra topics && uv run arboili transform news-topics` first.
Everything above runs on five fixture pages; everything below is the real thing.

In [9]:
if POOLED_PATH.exists():
    real_pooled = pl.read_parquet(POOLED_PATH)
    real_meta = pl.read_parquet(META_PATH)
    real_sweep = pl.read_parquet(SWEEP_PATH)

    print("=== the k sweep, on the real corpus ===")
    print(real_sweep.sort(TOPIC_ID))
    print()
    print("likelihood rises with k almost mechanically; coherence turns over.")
    print("That disagreement is the argument for the criterion, measured.")
    print()
    print(f"{real_pooled.height:,} rows = {real_pooled[[EW, LOCATION]].unique().height:,} "
          f"(ew, location) groups × {real_pooled[TOPIC_ID].n_unique()} topics")
    print(f"weeks {real_pooled[EW].n_unique():,} · locations {real_pooled[LOCATION].n_unique()}")
    print(f"model {real_pooled[MODEL_ID][0]}")
else:
    print("Run `uv run arboili transform news-topics` first")

=== the k sweep, on the real corpus ===
shape: (5, 5)
┌─────────────────────────┬──────────┬───────────┬─────────────┬──────────┐
│ model_id                ┆ topic_id ┆ coherence ┆ ll_per_word ┆ selected │
│ ---                     ┆ ---      ┆ ---       ┆ ---         ┆ ---      │
│ str                     ┆ i64      ┆ f32       ┆ f32         ┆ bool     │
╞═════════════════════════╪══════════╪═══════════╪═════════════╪══════════╡
│ tomotopy-dmr-k5-seed42  ┆ 5        ┆ 0.714133  ┆ -8.118479   ┆ false    │
│ tomotopy-dmr-k10-seed42 ┆ 10       ┆ 0.716466  ┆ -8.113792   ┆ false    │
│ tomotopy-dmr-k15-seed42 ┆ 15       ┆ 0.751039  ┆ -8.112279   ┆ true     │
│ tomotopy-dmr-k20-seed42 ┆ 20       ┆ 0.726279  ┆ -8.106872   ┆ false    │
│ tomotopy-dmr-k25-seed42 ┆ 25       ┆ 0.701593  ┆ -8.09151    ┆ false    │
└─────────────────────────┴──────────┴───────────┴─────────────┴──────────┘

likelihood rises with k almost mechanically; coherence turns over.
That disagreement is the argument for the 

In [10]:
if POOLED_PATH.exists():
    print("=== what the 15 topics are (mean share over all 965 weeks) ===")
    for topic_id in range(real_meta[TOPIC_ID].n_unique()):
        terms = (real_meta.filter(pl.col(TOPIC_ID) == topic_id)
                 .sort(WEIGHT, descending=True)[TERM].to_list())
        share = (real_pooled.filter((pl.col(TOPIC_ID) == topic_id) & (pl.col(LOCATION) == "BR"))
                 [SHARE].mean())
        print(f"  {topic_id:2d} (mean {share:.3f}): {', '.join(terms[:9])}")
    print()
    print("=== integrity over the real tables ===")
    for check in topic_checks(real_pooled, real_meta, real_sweep, FU_PATH):
        print(check)
else:
    print("Run `uv run arboili transform news-topics` first")

=== what the 15 topics are (mean share over all 965 weeks) ===
   0 (mean 0.038): brasil, dia, anos, campanha, educacao, escolas, rio, escola, criancas
   1 (mean 0.030): the, and, that, for, dengue, with, health, will, brazil
   2 (mean 0.091): mosquito, dengue, aedes, aegypti, combate, focos, agentes, populacao, transmissor
   3 (mean 0.040): zika, virus, chikungunya, febre, microcefalia, aedes, saude, amarela, casos
   4 (mean 0.054): governo, federal, presidente, publica, recursos, milhoes, ministerio, estado, nacional
   5 (mean 0.056): virus, dengue, pesquisa, fiocruz, pesquisadores, mosquitos, fundacao, testes, instituto
   6 (mean 0.075): rio, saude, janeiro, dengue, estado, secretaria, atendimento, cidade, estadual
   7 (mean 0.090): mais, gente, pessoas, fazer, temos, brasil, estamos, pais, anos
   8 (mean 0.072): saude, brasil, doencas, paises, milhoes, desenvolvimento, pais, mundial, malaria
   9 (mean 0.029): agua, pneus, ambiente, meio, regiao, ambiental, lixo, brasil, br

[PASS] share_schema: 6 columns in published order
[PASS] meta_schema: 4 columns in published order
[PASS] sweep_schema: 5 columns in published order
[PASS] share_unique_keys: 77,280 rows, all keys distinct
[PASS] meta_unique_keys: 225 rows, all keys distinct
[PASS] weeks_start_on_sunday: all ew dates are Sundays
[PASS] locations_known: 28 known locations
[PASS] shares_sum_to_one: 5,152 (ew, location) groups, max deviation 2.38e-07
[PASS] shares_in_unit_interval: every share is in [0, 1]
[PASS] finite: all shares and weights are finite
[PASS] article_counts: every row pools at least one article
[PASS] topic_ids_consistent: 15 topics, ids 0..14
[PASS] single_model: all published rows from tomotopy-dmr-k15-seed42
[PASS] sweep_selects_one: k=15 selected on coherence (0.7510; ll_per_word -8.1123)
[PASS] national_row_per_week: 965 weeks all carry BR


## Exploratory — does topic share track SINAN?

**This is not the protocol-00 comparison.** It is a first look at the paper's actual
question, and it carries the same caveats braço 01's and braço 02's notebooks carry:
coverage *follows* the epidemic curve, the news series is national while the target is
UF, and per-UF rows are mention-based.

The difference from braço 02 is that a topic share is a **scalar per topic**, so
there is nothing to reduce — and the correlation can be printed next to the topic's
own terms. That is the point of this braço: "topic 11 correlates 0.335" is checkable,
"PC1 correlates 0.119" is not.

SINAN here is `ew_symptom_onset`, the same Sunday calendar, summed to the week.

In [11]:
sinan_weekly = (
    pl.scan_parquet(SINAN_PATH)
    .group_by("ew_symptom_onset")
    .agg(pl.col("case_count").sum().alias("casos"))
    .collect()
)

if POOLED_PATH.exists():
    br = (
        real_pooled.filter(pl.col(LOCATION) == "BR")
        .join(sinan_weekly.rename({"ew_symptom_onset": EW}), on=EW, how="inner")
        .sort(EW)
    )
    print(f"national weeks with both series: {br[EW].n_unique():,}  "
          f"({br[EW].min()} → {br[EW].max()})\n")

    rows = []
    for topic_id in range(real_pooled[TOPIC_ID].n_unique()):
        group = br.filter(pl.col(TOPIC_ID) == topic_id)
        rows.append({
            "topic": topic_id,
            "rho vs cases": group.select(pl.corr(SHARE, "casos", method="spearman")).item(),
            "mean share": group[SHARE].mean(),
            "top terms": ", ".join(
                real_meta.filter(pl.col(TOPIC_ID) == topic_id)
                .sort(WEIGHT, descending=True)[TERM].to_list()[:6]
            ),
        })

    per_topic = (
        pl.DataFrame(rows)
        .with_columns(pl.col("rho vs cases").round(3), pl.col("mean share").round(3))
        .sort("rho vs cases", descending=True)
    )
    print(per_topic)
    print()
    print(f"for comparison, n_articles (braço 01's news_volume) reaches "
          f"{br.select(pl.corr(N_ARTICLES, 'casos', method='spearman')).item():.3f}")
else:
    print("Run `uv run arboili transform news-topics` first")

national weeks with both series: 481  (2013-07-28 → 2026-08-23)

shape: (15, 4)
┌───────┬──────────────┬────────────┬────────────────────────────────────────────────────────────┐
│ topic ┆ rho vs cases ┆ mean share ┆ top terms                                                  │
│ ---   ┆ ---          ┆ ---        ┆ ---                                                        │
│ i64   ┆ f64          ┆ f64        ┆ str                                                        │
╞═══════╪══════════════╪════════════╪════════════════════════════════════════════════════════════╡
│ 11    ┆ 0.335        ┆ 0.146      ┆ casos, dengue, ano, doenca, estado, numero                 │
│ 14    ┆ 0.205        ┆ 0.078      ┆ doenca, sintomas, febre, saude, dor, casos                 │
│ 6     ┆ 0.187        ┆ 0.043      ┆ rio, saude, janeiro, dengue, estado, secretaria            │
│ 2     ┆ 0.164        ┆ 0.074      ┆ mosquito, dengue, aedes, aegypti, combate, focos           │
│ 10    ┆ 0.057        ┆ 0.08

### Lead or lag?

If topic shares merely *follow* the epidemic curve, they are a lagging indicator and
useless for nowcasting. The scan shifts the case series backwards and re-correlates:
a peak at `k > 0` means the topic leads.

In [12]:
if POOLED_PATH.exists():
    top_topic = int(per_topic["topic"][0])
    group = br.filter(pl.col(TOPIC_ID) == top_topic)

    scan = []
    for k in range(0, 9):
        shifted = group.with_columns(pl.col("casos").shift(-k).alias("casos_futuros"))
        scan.append({
            "k (weeks)": k,
            "share[t] × cases[t+k]": shifted.select(
                pl.corr(SHARE, "casos_futuros", method="spearman")).item(),
        })
    print(f"lead/lag for topic {top_topic} "
          f"({', '.join(real_meta.filter(pl.col(TOPIC_ID) == top_topic).sort(WEIGHT, descending=True)[TERM].to_list()[:5])})\n")
    print(pl.DataFrame(scan).with_columns(pl.exclude("k (weeks)").round(3)))
else:
    print("Run `uv run arboili transform news-topics` first")

lead/lag for topic 11 (casos, dengue, ano, doenca, estado)

shape: (9, 2)
┌───────────┬───────────────────────┐
│ k (weeks) ┆ share[t] × cases[t+k] │
│ ---       ┆ ---                   │
│ i64       ┆ f64                   │
╞═══════════╪═══════════════════════╡
│ 0         ┆ 0.335                 │
│ 1         ┆ 0.325                 │
│ 2         ┆ 0.311                 │
│ 3         ┆ 0.28                  │
│ 4         ┆ 0.268                 │
│ 5         ┆ 0.23                  │
│ 6         ┆ 0.195                 │
│ 7         ┆ 0.165                 │
│ 8         ┆ 0.125                 │
└───────────┴───────────────────────┘


## What the exploratory section shows

Run on 2026-10-13, over the **481 national weeks** where both series exist
(2013-07-28 → 2026-08-23) — the same 481 weeks braço 02's notebook reads. Four
readings.

- **The topics are readable, and that is the deliverable.** Topic 11
  (*casos, dengue, ano, doença, estado, mortes*) is the surveillance-reporting
  register; topic 2 (*mosquito, aedes, aegypti, focos, água*) is vector control;
  topic 3 (*zika, chikungunya, microcefalia, febre amarela*) is the arbovirus-family
  topic; topic 10 (*vacina, doses, butantan*) is vaccination; topic 14 (*sintomas,
  febre, dor, tratamento*) is clinical presentation. A reader can say what each one
  is, which is the thing a 768-wide vector could never offer.
- **The best topic barely beats volume.** Topic 11 reaches **+0.335** against weekly
  cases; `n_articles` — braço 01's `news_volume` under another name — reaches
  **+0.347**. The best of fifteen topics does not out-correlate the count of
  articles in the week. That is the same uncomfortable reading braço 02's PC1 gave,
  arrived at from a completely different representation, and it is a stronger result
  for being replicated across two independent methods.
- **The spread across topics is large and interpretable.** It runs from **+0.335**
  (topic 11) down through the vaccine and vector topics in the middle to **−0.166**
  for topic 1. A topic model at least *tells you which* direction is which, instead
  of hiding it in a projection.
- **Topic 1 is the English-language artifact.** Its terms are *the, and, that, for,
  with, health, will* — Agência Brasil's English edition, 180 articles (4.1% of the
  kept corpus) that `lang_guess` labels `pt` because the detector only separates
  Spanish from not-Spanish. Its **−0.166** is a *negative* correlation with dengue
  cases, which is exactly what a non-Portuguese topic should show. This braço is the
  first representation to make that slice visible — braços 01 and 02 absorbed it
  silently — and it is **not fixed here**: protocol 00 fixes the article set as
  common ground, so re-gating it in one braço would break the comparison. It is an
  open item for protocol 00.

- **Contemporaneous, not leading.** Topic 11 peaks at `k = 0` (0.335) and decays
  monotonically to 0.125 at `k = 8`. Nothing here warns ahead. That matches braços
  01 and 02 and the framing decision in [`notes/README.md`](../notes/README.md):
  coverage measures attention, not the epidemic's future.

None of this is the protocol-00 comparison. That comparison — braços 01, 02 and 03
under one model and one set of metrics, with the negative control — has not run, and
these numbers must not be quoted as its result.

## Running it for real

This notebook mirrors `transform()` in
`src/pipeline/news/topics/transform.py`:

```bash
uv sync --extra topics                    # tomotopy (~30 MB), once
uv run arboili transform news-topics      # 4,369 articles → three tables
```

Writes `data/pipeline/news/EBC_dengue_topics_EW.parquet` (the weekly shares the
forecaster consumes), `..._topics_meta.parquet` (what each topic *is*) and
`..._topics_sweep.parquet` (the evidence for the k both were built at). Parquet plus
`csv.gz`, unlike braço 02: these tables are long and narrow with no dense vector
column, so they survive csv text form.

The run takes ~221 s on CPU at the default 1,000 iterations, single worker.

## What this notebook does not do

- **No rolling-origin refit.** DMR is fit once over the whole corpus, so the topics
  are shaped by all 24 years at once — including 2024, the record year. `notes/03`
  names this risk; the sweep makes the *selection* visible but does not address it.
- **No stability check across seeds.** One fit at one seed produced the published
  topics, and a single fit does not by itself establish that they are stable.
  Re-running the sweep at several seeds and reporting topic-to-topic agreement is the
  measurement that would settle it, and it has not been run.
- **No comparison against braços 01 and 02.** They publish the same grain so the
  comparison is possible; running it is protocol 00, and it has not run.
- **No negative control.** Shuffling the text and re-running is what protocol 00
  requires to prove the representation is being used at all — Tan 2024
  (arXiv:2406.16964) is the reason that control exists.
- **No fix for the English slice.** It is a corpus-level issue shared by all three
  braços, so it belongs to protocol 00 and not to this stage.